# 01 · Retrieval for RAG: chunking, BM25, embeddings, hybrid search with RRF, re-ranking, evaluation

**Goal:** build and *measure* the retrieval half of RAG from scratch, so you can explain every design choice
with numbers. **Time:** ~3 hours.

The corpus is a tiny fictional company handbook with labelled queries, so you can see each metric move.
Dense retrieval uses `sentence-transformers/all-MiniLM-L6-v2` (downloads ~90 MB once). With no internet, it
falls back to TF-IDF + SVD so the notebook still runs.

In [1]:
import os, re, math
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
import numpy as np, pandas as pd
from rank_bm25 import BM25Okapi

## 1. Corpus and labelled queries

In [2]:
docs = {
    "leave-1": "Employees get 20 days of paid annual leave per year. Unused leave up to 5 days can be carried over to next year.",
    "leave-2": "Sick leave: up to 14 days per year with a medical certificate for absences longer than 2 days.",
    "leave-3": "Parental leave is 16 weeks for the primary caregiver and 4 weeks for the secondary caregiver.",
    "remote-1": "Staff may work remotely up to 2 days a week after their probation period ends.",
    "remote-2": "Remote work requires a company laptop with full-disk encryption and the VPN switched on.",
    "it-1": "Reset your password from the self-service portal. Passwords expire every 90 days.",
    "it-2": "Report a lost or stolen laptop to the IT helpdesk within 24 hours so the device can be wiped.",
    "it-3": "The VPN client must be updated monthly; old versions are blocked automatically.",
    "pay-1": "Salaries are paid on the last working day of each month by bank transfer.",
    "pay-2": "Festival bonuses are paid twice a year, each equal to one month of basic salary.",
    "pay-3": "Expense claims must be submitted with receipts within 30 days of the purchase.",
    "train-1": "Each engineer has an annual learning budget of 30,000 BDT for courses, books and certifications.",
    "train-2": "Certification exam fees are reimbursed only if the exam is passed.",
    "sec-1": "Never paste client source code or credentials into public AI chatbots or online tools.",
    "sec-2": "Client data must stay on approved servers; copying it to personal devices is forbidden.",
    "onb-1": "New joiners complete a 3-month probation with a review at the end of each month.",
    "onb-2": "On day one, new joiners receive a laptop, an email account and access to the HR portal.",
    "health-1": "Medical insurance covers the employee, spouse and up to two children.",
    "health-2": "The office provides free lunch, snacks and a gym membership discount.",
    "travel-1": "International client visits need manager approval two weeks in advance.",
}
ids = list(docs); texts = [docs[i] for i in ids]

queries = {
    "How many vacation days do I get each year?": {"leave-1"},
    "Can I carry unused holidays into next year?": {"leave-1"},
    "maternity or paternity leave length": {"leave-3"},
    "Can I work from home?": {"remote-1", "remote-2"},
    "I forgot my password": {"it-1"},
    "my laptop was stolen": {"it-2"},
    "VPN rules": {"remote-2", "it-3"},
    "when is salary paid": {"pay-1"},
    "how much is the Eid bonus": {"pay-2"},
    "budget for online courses": {"train-1", "train-2"},
    "is it ok to use ChatGPT with client code": {"sec-1"},
    "health insurance for my family": {"health-1"},
}
print(len(docs), "documents |", len(queries), "labelled queries")

20 documents | 12 labelled queries


## 2. Chunking (real documents are long)
Smaller chunks → precise matches but lost context; bigger chunks → more context but diluted embeddings.
Overlap stops an answer being cut in half. Typical start: 200–500 tokens, 10–20% overlap; then **measure**.

In [3]:
def chunk_words(text, size=40, overlap=10):
    words = text.split(); step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, max(len(words) - overlap, 1), step)]

long_text = " ".join(texts)
chunks = chunk_words(long_text, size=40, overlap=10)
print(len(long_text.split()), "words ->", len(chunks), "chunks; first chunk:\n", chunks[0])

289 words -> 10 chunks; first chunk:
 Employees get 20 days of paid annual leave per year. Unused leave up to 5 days can be carried over to next year. Sick leave: up to 14 days per year with a medical certificate for absences longer than 2


## 3. Sparse retrieval: BM25
BM25 scores term matches, weighted by rarity (IDF), with saturation for repeated terms (k1) and document
length normalisation (b). Great for exact terms: names, codes, "VPN".

In [4]:
tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())
bm25 = BM25Okapi([tok(t) for t in texts])

def search_bm25(q, k=5):
    s = bm25.get_scores(tok(q))
    return [ids[i] for i in np.argsort(-s)[:k]]
print(search_bm25("How many vacation days do I get each year?"))

['leave-1', 'pay-2', 'leave-2', 'pay-1', 'onb-1']


## 4. Dense retrieval: embeddings + cosine similarity
A bi-encoder embeds query and documents **separately**, so document vectors can be pre-computed and
indexed (FAISS, pgvector, Qdrant). Great for paraphrases ("vacation" ≈ "annual leave").

In [5]:
try:
    from sentence_transformers import SentenceTransformer
    enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embed = lambda xs: enc.encode(xs, normalize_embeddings=True)
    DENSE = "all-MiniLM-L6-v2"
except Exception as e:                      # offline fallback
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import Normalizer
    lsa = make_pipeline(TfidfVectorizer(), TruncatedSVD(12, random_state=0), Normalizer()).fit(texts)
    embed = lambda xs: lsa.transform(xs)
    DENSE = f"TF-IDF+SVD fallback ({type(e).__name__})"
doc_vecs = embed(texts)
print("dense model:", DENSE, "| vectors:", doc_vecs.shape)

def search_dense(q, k=5):
    s = doc_vecs @ embed([q])[0]           # vectors are normalised -> dot = cosine
    return [ids[i] for i in np.argsort(-s)[:k]]
print(search_dense("How many vacation days do I get each year?"))

dense model: all-MiniLM-L6-v2 | vectors: (20, 384)
['leave-1', 'leave-2', 'pay-2', 'leave-3', 'pay-1']


## 5. Hybrid search with Reciprocal Rank Fusion (RRF)
BM25 and cosine scores live on different scales, so fuse **ranks**, not scores:

RRF(d) = Σ over retrievers 1 / (k + rank(d)), with k = 60 (Cormack et al., 2009).

In [6]:
def rrf(rank_lists, k=60, top=5):
    scores = {}
    for ranking in rank_lists:
        for r, d in enumerate(ranking, start=1):
            scores[d] = scores.get(d, 0) + 1 / (k + r)
    return sorted(scores, key=scores.get, reverse=True)[:top]

def search_hybrid(q, k=5, depth=10):
    return rrf([search_bm25(q, depth), search_dense(q, depth)], top=k)
print(search_hybrid("VPN rules"))

['it-3', 'remote-2', 'remote-1', 'pay-2', 'leave-1']


## 6. Evaluation metrics, implemented from scratch
- **Recall@k**: share of relevant docs found in the top k.
- **MRR**: 1 / rank of the first relevant doc, averaged over queries.
- **nDCG@k**: rewards relevant docs near the top; 1.0 = perfect order.

In [7]:
def recall_at_k(ranked, rel, k): return len(set(ranked[:k]) & rel) / len(rel)
def mrr(ranked, rel): return next((1 / r for r, d in enumerate(ranked, 1) if d in rel), 0.0)
def ndcg_at_k(ranked, rel, k):
    dcg = sum(1 / math.log2(r + 1) for r, d in enumerate(ranked[:k], 1) if d in rel)
    ideal = sum(1 / math.log2(r + 1) for r in range(1, min(len(rel), k) + 1))
    return dcg / ideal

def evaluate(search, k=3):
    rows = [(recall_at_k(search(q, 10), rel, 1), recall_at_k(search(q, 10), rel, k),
             mrr(search(q, 10), rel), ndcg_at_k(search(q, 10), rel, k)) for q, rel in queries.items()]
    return np.mean(rows, axis=0)

table = pd.DataFrame({name: evaluate(fn) for name, fn in
                      {"BM25": search_bm25, f"Dense": search_dense, "Hybrid (RRF)": search_hybrid}.items()},
                     index=["Recall@1", "Recall@3", "MRR", "nDCG@3"]).T
table.round(3)

,Recall@1,Recall@3,MRR,nDCG@3
BM25,0.500,0.792,0.715,0.703
Dense,0.875,1.000,1.000,1.000
Hybrid (RRF),0.625,0.875,0.847,0.823


**Read this honestly.** On this tiny, paraphrase-heavy corpus, dense retrieval wins and plain RRF *dilutes*
it, because BM25 adds noise when queries share no words with the answer ("Eid bonus" vs "festival bonuses").
Hybrid search pays off when exact terms matter: product codes, names, error messages, legal clause numbers.
The lesson is not "hybrid is always better": **measure on your own labelled queries**, then try weighted RRF
or a re-ranker (section 7).

In [8]:
# where does each method fail? (error analysis beats staring at averages)
for q, rel in queries.items():
    b, d = search_bm25(q, 3), search_dense(q, 3)
    if not (set(b) & rel) or not (set(d) & rel):
        print(f"{q!r}\n   relevant={rel}\n   BM25 top3={b}\n   dense top3={d}")

'maternity or paternity leave length'
   relevant={'leave-3'}
   BM25 top3=['sec-1', 'leave-1', 'it-2']
   dense top3=['leave-3', 'leave-2', 'leave-1']
'how much is the Eid bonus'
   relevant={'pay-2'}
   BM25 top3=['train-2', 'leave-3', 'sec-2']
   dense top3=['pay-2', 'onb-2', 'train-2']


## 7. Re-ranking with a cross-encoder (optional, downloads ~90 MB)
A cross-encoder reads query and document **together**, so it is more accurate but too slow to run over
the whole corpus. Pattern: retrieve top 20–50 cheaply → re-rank → keep top 3–5 for the LLM prompt.

In [9]:
try:
    from sentence_transformers import CrossEncoder
    ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    def search_rerank(q, k=5, depth=10):
        cand = search_hybrid(q, depth, depth)
        s = ce.predict([(q, docs[c]) for c in cand])
        return [cand[i] for i in np.argsort(-s)[:k]]
    table.loc["Hybrid + cross-encoder"] = evaluate(search_rerank)
except Exception as e:
    print("cross-encoder skipped:", type(e).__name__)
table.round(3)

,Recall@1,Recall@3,MRR,nDCG@3
BM25,0.500,0.792,0.715,0.703
Dense,0.875,1.000,1.000,1.000
Hybrid (RRF),0.625,0.875,0.847,0.823
Hybrid + cross-encoder,0.875,0.958,1.000,0.968


## 8. From retrieval to a grounded prompt
Number the evidence, demand citations, and give the model a way to say "I don't know".

In [10]:
def build_prompt(question, k=3):
    hits = search_hybrid(question, k)
    evidence = "\n".join(f"[{i}] ({h}) {docs[h]}" for i, h in enumerate(hits, 1))
    return (
        "Answer using ONLY the evidence below. Cite evidence numbers like [1]. "
        "If the evidence does not contain the answer, reply exactly: I don't know.\n\n"
        f"Evidence:\n{evidence}\n\nQuestion: {question}\nAnswer:")
print(build_prompt("Can I carry unused holidays into next year?"))

Answer using ONLY the evidence below. Cite evidence numbers like [1]. If the evidence does not contain the answer, reply exactly: I don't know.

Evidence:
[1] (leave-1) Employees get 20 days of paid annual leave per year. Unused leave up to 5 days can be carried over to next year.
[2] (leave-2) Sick leave: up to 14 days per year with a medical certificate for absences longer than 2 days.
[3] (pay-2) Festival bonuses are paid twice a year, each equal to one month of basic salary.

Question: Can I carry unused holidays into next year?
Answer:


Send this prompt to any LLM (OpenAI, Claude, Gemini, or a local vLLM server via its OpenAI-compatible API).
Then evaluate the **generation** half too: faithfulness (is every claim supported by the cited evidence?)
and answer relevance. Tools: RAGAS, or LLM-as-judge with a rubric, checked against human labels.

## ✍️ Your turn
1. Add 5 harder queries (typos, synonyms, two-hop questions). Which retriever breaks first?
2. Tune RRF: try k = 10, 60, 100, and weight BM25 twice as much. Report the table.
3. Re-build this pipeline in **LlamaIndex** (VectorStoreIndex + BM25Retriever + QueryFusionRetriever)
   and compare the numbers and lines of code with this version.
4. Store the vectors in FAISS (`faiss.IndexFlatIP`) and check you get the same ranking.

## 🎯 Interview check
- Why hybrid search? Give a query where BM25 wins and one where dense wins.
- Bi-encoder vs cross-encoder?
- How do you evaluate a RAG system end to end? What do you do with no labelled data?
- How do you reduce hallucination in RAG? How do you prove it worked?
- What is prompt injection in a RAG system, and one defence?

In [11]:
# Your code here